# Data cleaning and preparations

## TODO
1. [ ] Decide what to do with missing values (e.g. january 1-16 values in 1_tahe)
2. [x] Decide whether to incorporate 2026 as well. Where and how.
3. [x] Decide whether to incorporate recently added signs from after 2023.
4. [ ] Rename filename column to sign label or something obvious.
5. [ ] Decide if and how to organize getting data for 2021-2024.

## Installation
* install python duckdb for anaconda (version 1.5.6 specifically, not the newest one)
* create a motherduck account and join the organization from link sent to UT email
* generate a motherduck read-write token for yourself under the sandra_md org and add it to anaconda env variables

In [ ]:
import glob
import pandas as pd
from pathlib import Path
import duckdb

## Creating a database
The following code was used to generate a DuckDB database from the data sent to us by the Tartu city government and upload it to MotherDuck for use without downloading. This was done due to the large volume of data (76 million rows) and API (1000 rows/request) and CSV download (10 000 rows/download) limits. These limits were the reason we requested the data separately instead of downloading it ourselves. NB! The council only sent us data from 2025–2026 (34 million rows total). This code was created with help from AI (Claude).

```
print(duckdb.__version__) # check it's 1.5.6
```

```
folder = Path("data/Kiirustablood_arhiiv_2")

for f in folder.rglob("*.csv"): # rglob also searches in nested folders
    df = pd.read_csv(f, sep=";")
    df["filename"] = f.stem        # copy the file's name to a new column for each row, no .csv extension
    # this is because the file name is the label (tunnus) of the speed sign, which we need (the file itself doesn't contain it)
    df.to_csv(f, sep=";", index=False)  
```

```
con = duckdb.connect("kiirustablood_2526_new.duckdb") # connecting to local DuckDB

con.execute("""
    CREATE OR REPLACE TABLE all_data AS
    SELECT *
    FROM read_csv(
        'data/Kiirustablood_arhiiv_2/**/*.csv',
        delim = ';',
        header = true,
        union_by_name = true
    )
""")

print(con.execute("SELECT count(*) FROM all_data").fetchone()) # row count check
```

```
con = duckdb.connect("md:") # connects to MotherDuck remotely automatically using MOTHERDUCK_TOKEN from env variables or motherduck_token from anaconda
con.execute("CREATE DATABASE IF NOT EXISTS idsprojectdb")
con.execute("ATTACH 'kiirustablood_2526_new.duckdb' AS local_db (READ_ONLY)")

con.execute("""
    CREATE OR REPLACE TABLE idsprojectdb.main.all_data AS
    SELECT * FROM local_db.main.all_data
""")

print(con.execute("SELECT count(*) FROM idsprojectdb.main.all_data").fetchone()) # row count check (see if it matches local db row count)
```

```
con.execute("CREATE SHARE idsproject_share FROM idsprojectdb (ACCESS ORGANIZATION)")
```

```
# TEST to see if db works from online

con = duckdb.connect("md:")

df = con.execute("""
    SELECT *
    FROM idsprojectdb.main.all_data
    LIMIT 10
""").df()

print(df)
```

## Sharing a table you have created or attaching one by another person
### Use this to see all schemas shared with you   
*NB! Shares created by others are read-only. You need to create your own db with your work (you can use the contents of another person's db to create the data, but if you want to make and save changes, that's when the new db comes in).*
```
con = duckdb.connect("md:")
print(con.execute("SELECT * FROM md_information_schema.shared_with_me").df())
```

### Use this to share   
```
con.execute("""
    CREATE OR REPLACE SHARE sharei_nimi_mille_ise_praegu_valid FROM sinu_db_täpne_nimi_kuhu_tabeli_tegid
    (ACCESS ORGANIZATION, VISIBILITY DISCOVERABLE, UPDATE AUTOMATIC)
""")
print(con.execute("LIST SHARES").fetchall())
```
-> check that it appears  

Then on the MotherDuck website, find your share, click on it and click "attach",
then three dots -> "copy share link" -> paste it as a comment in Jupyter or share with others in some other way.

# ---------- OLD -------------

In [ ]:
# function for turning TV_kiirustabloo.csv into a dataframe and removing unnecessary columns
def process_sign_info_file(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename)
    df_sub = df[['X','Y','Tunnus','Maks_kiirus']] # kas tahame koordinaate alles hoida?
    return df_sub

In [ ]:
# 4 km/h is always subtracted from any speeding incidents (mõõtemääramatus)
# speed cameras register speedings starting from 7 km/h over the limit, mobile police radars from 10 km/h over
# currently default speeding tolerance is therefore 6 km/h, since it's the lowest you can get away with
# currently instances w/ speeds under 5 km/h and over 150 km/h are removed, we should assess if these bounds are suitable
# currently hours where less than 5 cars passed are removed, should assess if we should keep that
# weekday -> monday = 0
def process_measured_speeds_file(measured_speeds_file: str, sign_info_file="data/TV_kiirustabloo.csv", min_passing_cars=5, 
                                 speeding_tolerance_kmh=6, min_speed=5, max_speed=150) -> pd.DataFrame:
    df = pd.read_csv(
        measured_speeds_file,
        usecols=["Tunnus", "V1", "Year", "Month", "Day", "Hour"],
        dtype={"V1": "float32", "Tunnus": "category"},
    )

    # eemalda mõõtmisvead(?) ja äärmused
    df = df[df["V1"].between(min_speed, max_speed)]

    # lisa piirang ja märgi ületajate arv
    limits_df = process_sign_info_file(sign_info_file)
    df = df.merge(limits_df, on='Tunnus')
    df["Speeders_count"] = (df["V1"] > df["Maks_kiirus"] + speeding_tolerance_kmh).astype("int8")

    hour_stats_df = (
        df.groupby(["Tunnus", "Year", "Month", "Day", "Hour", "Maks_kiirus"], observed=True) # add "X","Y" to groupby to see coordinate columns
          .agg(passers_count=("V1", "size"),
               avg_speed=("V1", "mean"),
               speeders_count=("Speeders_count", "sum"))
          .reset_index()
    )

    hour_stats_df["speeders_share"] = hour_stats_df["speeders_count"] / hour_stats_df["passers_count"]
    hour_stats_df = hour_stats_df[hour_stats_df["passers_count"] >= min_passing_cars]

    # lisa nädalapäev
    kuupaev = pd.to_datetime(
        hour_stats_df[["Year", "Month", "Day"]].rename(columns=str.lower)
    )
    hour_stats_df["weekday"] = kuupaev.dt.dayofweek  # 0 = esmaspäev

    return hour_stats_df

In [ ]:
df = process_measured_speeds_file("data/2_ulikooli.csv")
df.head(15)